In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_telco_silver")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c8e52408-bc0e-4d75-9009-6a431843c961;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (724ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (79ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (119ms)
:: resolution report :: resolve 2051ms :: artifacts dl 9

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [4]:
path = "s3a://bronze/base_telco/"
df_base_telco = spark.read.parquet(path)
df_base_telco.cache()
df_base_telco.show(20, truncate=False)

26/01/01 13:36:22 WARN CacheManager: Asked to cache already cached data.


+-----------+------+---------------+---+----+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72|var_73|var_74|var_75|var_76|var_77|var_78|var_79|var_80|var_81|var_82|var_83|var_84|

In [5]:
df_base_telco.createOrReplaceTempView("raw_00")

In [7]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM raw_00
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(20, truncate=False)

+------+------------+-------------+
|SAFRA |total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|219860      |219860       |
|202411|240520      |240520       |
|202412|241453      |241453       |
|202501|233710      |233710       |
|202502|214665      |214665       |
|202503|216896      |216896       |
+------+------------+-------------+



In [9]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM raw_00
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM raw_00 r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [10]:
df_resultado = contagem_percentual("PROD")
df_resultado.show(truncate=False)

26/01/01 13:45:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/01 13:45:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/01 13:45:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/01 13:45:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/01 13:45:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/01 13:45:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/01 1

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|CMV         |1346917      |98.52        |
|NET         |16209        |1.19         |
|DTH         |3978         |0.29         |
+------------+-------------+-------------+



In [8]:
print('lista de colunas para tipar')
for col in spark.table("raw_00").columns:
    print('try_cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
try_cast(NUM_CPF as) as NUM_CPF,
try_cast(SAFRA as) as SAFRA,
try_cast(FLAG_INSTALACAO as) as FLAG_INSTALACAO,
try_cast(FPD as) as FPD,
try_cast(PROD as) as PROD,
try_cast(flag_mig2 as) as flag_mig2,
try_cast(var_26 as) as var_26,
try_cast(var_27 as) as var_27,
try_cast(var_28 as) as var_28,
try_cast(var_29 as) as var_29,
try_cast(var_30 as) as var_30,
try_cast(var_31 as) as var_31,
try_cast(var_32 as) as var_32,
try_cast(var_33 as) as var_33,
try_cast(var_34 as) as var_34,
try_cast(var_35 as) as var_35,
try_cast(var_36 as) as var_36,
try_cast(var_37 as) as var_37,
try_cast(var_38 as) as var_38,
try_cast(var_39 as) as var_39,
try_cast(var_40 as) as var_40,
try_cast(var_41 as) as var_41,
try_cast(var_42 as) as var_42,
try_cast(var_43 as) as var_43,
try_cast(var_44 as) as var_44,
try_cast(var_45 as) as var_45,
try_cast(var_46 as) as var_46,
try_cast(var_47 as) as var_47,
try_cast(var_48 as) as var_48,
try_cast(var_49 as) as var_49,
try_cast(var_50 as) as var_5

In [11]:
lake = spark.sql(     
    """
        select
        
            -- colunas do arquivo --

            try_cast(NUM_CPF as STRING) as NUM_CPF,
            try_cast(SAFRA as INT) as SAFRA,
            try_cast(FLAG_INSTALACAO as INT) as FLAG_INSTALACAO,
            try_cast(FPD as INT) as FPD,
            try_cast(PROD as STRING) as PROD,
            try_cast(flag_mig2 as STRING) as flag_mig2,
            try_cast(var_26 as DOUBLE) as var_26,
            try_cast(var_27 as DOUBLE) as var_27,
            try_cast(var_28 as DOUBLE) as var_28,
            try_cast(var_29 as DOUBLE) as var_29,
            try_cast(var_30 as DOUBLE) as var_30,
            try_cast(var_31 as DOUBLE) as var_31,
            try_cast(var_32 as DOUBLE) as var_32,
            try_cast(var_33 as DOUBLE) as var_33,
            try_cast(var_34 as DOUBLE) as var_34,
            try_cast(var_35 as DOUBLE) as var_35,
            try_cast(var_36 as DOUBLE) as var_36,
            try_cast(var_37 as DOUBLE) as var_37,
            try_cast(var_38 as DOUBLE) as var_38,
            try_cast(var_39 as DOUBLE) as var_39,
            try_cast(var_40 as DOUBLE) as var_40,
            try_cast(var_41 as DOUBLE) as var_41,
            try_cast(var_42 as DOUBLE) as var_42,
            try_cast(var_43 as DOUBLE) as var_43,
            try_cast(var_44 as DOUBLE) as var_44,
            try_cast(var_45 as DOUBLE) as var_45,
            try_cast(var_46 as DOUBLE) as var_46,
            try_cast(var_47 as DOUBLE) as var_47,
            try_cast(var_48 as DOUBLE) as var_48,
            try_cast(var_49 as DOUBLE) as var_49,
            try_cast(var_50 as DOUBLE) as var_50,
            try_cast(var_51 as DOUBLE) as var_51,
            try_cast(var_52 as DOUBLE) as var_52,
            try_cast(var_53 as DOUBLE) as var_53,
            try_cast(var_54 as DOUBLE) as var_54,
            try_cast(var_55 as DOUBLE) as var_55,
            try_cast(var_56 as DOUBLE) as var_56,
            try_cast(var_57 as DOUBLE) as var_57,
            try_cast(var_58 as DOUBLE) as var_58,
            try_cast(var_59 as DOUBLE) as var_59,
            try_cast(var_60 as DOUBLE) as var_60,
            try_cast(var_61 as DOUBLE) as var_61,
            try_cast(var_62 as DOUBLE) as var_62,
            try_cast(var_63 as DOUBLE) as var_63,
            try_cast(var_64 as DOUBLE) as var_64,
            try_cast(var_65 as DOUBLE) as var_65,
            try_cast(var_66 as DOUBLE) as var_66,
            try_cast(var_67 as DOUBLE) as var_67,
            try_cast(var_68 as DOUBLE) as var_68,
            try_cast(var_69 as DOUBLE) as var_69,
            try_cast(var_70 as DOUBLE) as var_70,
            try_cast(var_71 as DOUBLE) as var_71,
            try_cast(var_72 as DOUBLE) as var_72,
            try_cast(var_73 as DOUBLE) as var_73,
            try_cast(var_74 as DOUBLE) as var_74,
            try_cast(var_75 as DOUBLE) as var_75,
            try_cast(var_76 as DOUBLE) as var_76,
            try_cast(var_77 as DOUBLE) as var_77,
            try_cast(var_78 as DOUBLE) as var_78,
            try_cast(var_79 as DOUBLE) as var_79,
            try_cast(var_80 as DOUBLE) as var_80,
            try_cast(var_81 as DOUBLE) as var_81,
            try_cast(var_82 as DOUBLE) as var_82,
            try_cast(var_83 as DOUBLE) as var_83,
            try_cast(var_84 as DOUBLE) as var_84,
            try_cast(var_85 as DOUBLE) as var_85,
            try_cast(var_86 as DOUBLE) as var_86,
            try_cast(var_87 as DOUBLE) as var_87,
            try_cast(var_88 as DOUBLE) as var_88,
            try_cast(var_89 as DOUBLE) as var_89,
            try_cast(var_90 as DOUBLE) as var_90,
            try_cast(var_91 as DOUBLE) as var_91,
            try_cast(var_92 as DOUBLE) as var_92,
            try_cast(var_93 as DOUBLE) as var_93,
            {pdthproc} as DATPROC

        from
            raw_00
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

1367104

In [12]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- FLAG_INSTALACAO: integer (nullable = true)
 |-- FPD: integer (nullable = true)
 |-- PROD: string (nullable = true)
 |-- flag_mig2: string (nullable = true)
 |-- var_26: double (nullable = true)
 |-- var_27: double (nullable = true)
 |-- var_28: double (nullable = true)
 |-- var_29: double (nullable = true)
 |-- var_30: double (nullable = true)
 |-- var_31: double (nullable = true)
 |-- var_32: double (nullable = true)
 |-- var_33: double (nullable = true)
 |-- var_34: double (nullable = true)
 |-- var_35: double (nullable = true)
 |-- var_36: double (nullable = true)
 |-- var_37: double (nullable = true)
 |-- var_38: double (nullable = true)
 |-- var_39: double (nullable = true)
 |-- var_40: double (nullable = true)
 |-- var_41: double (nullable = true)
 |-- var_42: double (nullable = true)
 |-- var_43: double (nullable = true)
 |-- var_44: double (nullable = true)
 |-- var_45: double (nullable = tru

In [13]:
lake.show(5)

+-----------+------+---------------+---+----+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+--------------+
|    NUM_CPF| SAFRA|FLAG_INSTALACAO|FPD|PROD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72|var_73|var_74|var_75|var_76|var_77|var_78|var_79|var_80|var_81|var_82

In [14]:
for col in lake.columns:
    agg_result = lake.agg(
        {col: "count"} 
    ).collect()[0]
    
    total = lake.count()
    nao_nulos = agg_result[f"count({col})"]
    nulos = total - nao_nulos
    
    if nulos > 0:
        print(f"{col}: {nulos} nulos ({nulos/total*100:.2f}%)")

FPD: 45936 nulos (3.36%)
flag_mig2: 58130 nulos (4.25%)
var_26: 1295 nulos (0.09%)
var_27: 1295 nulos (0.09%)
var_28: 1295 nulos (0.09%)
var_29: 1295 nulos (0.09%)
var_30: 1295 nulos (0.09%)
var_31: 1295 nulos (0.09%)
var_32: 1295 nulos (0.09%)
var_33: 1295 nulos (0.09%)
var_34: 1295 nulos (0.09%)
var_35: 1295 nulos (0.09%)
var_36: 1295 nulos (0.09%)
var_37: 1295 nulos (0.09%)
var_38: 1295 nulos (0.09%)
var_39: 1295 nulos (0.09%)
var_40: 1295 nulos (0.09%)
var_41: 1295 nulos (0.09%)
var_42: 1295 nulos (0.09%)
var_43: 1295 nulos (0.09%)
var_44: 1295 nulos (0.09%)
var_45: 1295 nulos (0.09%)
var_46: 1295 nulos (0.09%)
var_47: 1295 nulos (0.09%)
var_48: 1295 nulos (0.09%)
var_49: 1295 nulos (0.09%)
var_50: 1295 nulos (0.09%)
var_51: 1295 nulos (0.09%)
var_52: 1295 nulos (0.09%)
var_53: 1295 nulos (0.09%)
var_54: 1295 nulos (0.09%)
var_55: 1295 nulos (0.09%)
var_56: 1295 nulos (0.09%)
var_57: 1295 nulos (0.09%)
var_58: 1295 nulos (0.09%)
var_59: 1295 nulos (0.09%)
var_60: 1295 nulos (0.09%)

In [17]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, SAFRA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup.cache()
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

1367104

In [18]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(20)

+------+------------+-------------+
| SAFRA|total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|      219860|       219860|
|202411|      240520|       240520|
|202412|      241453|       241453|
|202501|      233710|       233710|
|202502|      214665|       214665|
|202503|      216896|       216896|
+------+------------+-------------+



In [19]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_telco/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )
    print("Dados inseridos com sucesso...")

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Dados inseridos com sucesso...")

Tabela silver não existe. Criando...


In [26]:
name = "base_telco"

df_controle = spark.sql("""
    SELECT
        '{name_table}'        AS nome_tabela,
        SAFRA                 AS safra,
        COUNT(*)              AS qtd_registros,
        current_timestamp()   AS datproc
    FROM lake_dedup
    GROUP BY SAFRA
""".format(name_table=name))

df_controle.show()


+-----------+------+-------------+--------------------+
|nome_tabela| safra|qtd_registros|             datproc|
+-----------+------+-------------+--------------------+
| base_telco|202501|       233710|2026-01-01 13:55:...|
| base_telco|202412|       241453|2026-01-01 13:55:...|
| base_telco|202502|       214665|2026-01-01 13:55:...|
| base_telco|202503|       216896|2026-01-01 13:55:...|
| base_telco|202411|       240520|2026-01-01 13:55:...|
| base_telco|202410|       219860|2026-01-01 13:55:...|
+-----------+------+-------------+--------------------+



In [27]:
silver_controle_path = "s3a://silver/controle/"
if not DeltaTable.isDeltaTable(spark, silver_controle_path):
    print("Tabela de controle não existe. Criando...")

    (
        df_controle
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")
else:
    print("Tabela de controle existe. Inserindo novo registro...")

    (
        df_controle
        .write
        .format("delta")
        .mode("append")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")

Tabela de controle não existe. Criando...


In [28]:
spark.stop()